[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gad-gg-23/Identificador_Rostros_Modelo/blob/main/colab/Entrenamiento_Modelo.ipynb)

# Entrenamiento del identificador de rostros

Corre esto en un runtime con GPU: **Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU**.

In [ ]:
!git clone https://github.com/Gad-gg-23/Identificador_Rostros_Modelo.git
%cd Identificador_Rostros_Modelo
!pip install -q -r requirements.txt

## Login a Hugging Face

Necesitas una cuenta agregada como miembro de la organizacion `escom-identificador-rostros`.

In [ ]:
from huggingface_hub import login
login()

## 1. Descargar el dataset

In [ ]:
!python descargar_datos.py

## 2. Montar Google Drive (para no perder el progreso si Colab se desconecta)

Los rostros preprocesados y los checkpoints de entrenamiento se guardaran en tu Drive.
Si la sesion se cae, vuelve a correr desde esta celda y luego usa `--resume` al entrenar.

## 3. Detectar y alinear rostros (MTCNN)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import yaml
from pathlib import Path

carpeta_drive = Path('/content/drive/MyDrive/ESCOM_ML_Identificador_Rostros')
carpeta_drive.mkdir(parents=True, exist_ok=True)

with open('config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['datos']['rostros_raiz'] = str(carpeta_drive / 'rostros')
cfg['salida']['checkpoints'] = str(carpeta_drive / 'checkpoints')
cfg['salida']['mejor_modelo'] = str(carpeta_drive / 'checkpoints' / 'mejor_modelo.pt')
cfg['salida']['log_csv'] = str(carpeta_drive / 'checkpoints' / 'historial_entrenamiento.csv')

with open('config.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)

print('rostros/ y checkpoints/ ahora viven en Google Drive:', carpeta_drive)

In [ ]:
!python src/preprocesar_rostros.py --config config.yaml

## 4. Entrenar (2 fases: cabeza ArcFace, luego fine-tuning)

In [ ]:
!python src/entrenar.py --config config.yaml --resume

## 5. Evaluar

In [ ]:
!python src/evaluar.py --config config.yaml

## 6. Descargar el checkpoint entrenado

In [ ]:
import yaml
from google.colab import files

with open('config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

files.download(cfg['salida']['mejor_modelo'])

## 7. Probar una foto nueva

In [ ]:
from google.colab import files
subido = files.upload()
nombre_foto = list(subido.keys())[0]
!python src/inferir.py --imagen {nombre_foto} --config config.yaml